In [ ]:
# Install dependencies
!pip install redis fastapi uvicorn mlxtend nest_asyncio
!apt-get install redis-server > /dev/null
!redis-server --daemonize yes

In [ ]:
import logging
import warnings

# Hide the background noise
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("uvicorn.access").setLevel(logging.ERROR)

In [ ]:
import pandas as pd
import random

def generate_data():
    items_map = {
        'Main': ['Chicken Biryani', 'Margherita Pizza', 'Masala Dosa', 'Veg Burger'],
        'Side': ['Raita', 'Salan', 'Garlic Bread', 'Fries'],
        'Beverage': ['Coke', 'Filter Coffee', 'Thums Up', 'Lassi'],
        'Dessert': ['Gulab Jamun', 'Brownie']
    }

    data = []
    for i in range(1, 501):
        order_id = 1000 + i
        user_id = f"U{random.randint(100, 999)}"
        time = random.choice(['Morning', 'Lunch', 'Evening', 'Late-night'])


        if time == 'Morning': main = 'Masala Dosa'
        elif time == 'Lunch': main = 'Chicken Biryani'
        else: main = random.choice(['Margherita Pizza', 'Veg Burger'])

        data.append([order_id, user_id, main, 'Main', 1, time])

        # Add pairings (The signal for Apriori)
        if main == 'Chicken Biryani':
            data.append([order_id, user_id, random.choice(['Raita', 'Salan']), 'Side', 1, time])
        if main == 'Margherita Pizza':
            data.append([order_id, user_id, 'Garlic Bread', 'Side', 1, time])

    return pd.DataFrame(data, columns=['Order_ID', 'User_ID', 'Item_Name', 'Category', 'Quantity', 'Time_of_Day'])

df_synthetic = generate_data()
print("Synthetic Dataset Created!")

Synthetic Dataset Created!


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
import json

def run_apriori(df):
    basket = (df.groupby(['Order_ID', 'Item_Name'])['Quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('Order_ID'))
    basket_sets = basket.map(lambda x: 1 if x > 0 else 0)

    freq_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)
    rules = association_rules(freq_itemsets, metric="lift", min_threshold=1.2)

    mapping = {}
    for _, row in rules.iterrows():
        ant = list(row['antecedents'])[0]
        con = list(row['consequents'])[0]
        if ant not in mapping: mapping[ant] = []
        mapping[ant].append({"item": con, "score": row['lift']})

    with open('apriori_rules.json', 'w') as f:
        json.dump(mapping, f)
    return mapping

rules_mapping = run_apriori(df_synthetic)
print("Apriori Rules saved to apriori_rules.json")

Apriori Rules saved to apriori_rules.json


In [ ]:
import torch
import torch.nn as nn

class BSTModel(nn.Module):
    def __init__(self, vocab_size=100, embed_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True), num_layers=2
        )
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.transformer(self.embedding(x))
        return self.fc(x[:, -1, :])

# Save placeholder model
model = BSTModel()
torch.save(model, "bst_model.pt")

# Create a Name-to-ID mapping for the Transformer
all_items = df_synthetic['Item_Name'].unique()
item_to_id = {name: i for i, name in enumerate(all_items)}
id_to_name = {i: name for name, i in item_to_id.items()}

with open('item_map.json', 'w') as f:
    json.dump(item_to_id, f)

In [ ]:
import torch
import torch.nn as nn

class BSTModel(nn.Module):
    def __init__(self, vocab_size=100, embed_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True), num_layers=2
        )
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.transformer(self.embedding(x))
        return self.fc(x[:, -1, :])

# Save placeholder model
model = BSTModel()
torch.save(model, "bst_model.pt")

# Create a Name-to-ID mapping for the Transformer
all_items = df_synthetic['Item_Name'].unique()
item_to_id = {name: i for i, name in enumerate(all_items)}
id_to_name = {i: name for name, i in item_to_id.items()}

with open('item_map.json', 'w') as f:
    json.dump(item_to_id, f)

In [ ]:
import redis

r = redis.Redis(host='localhost', port=6379, db=0)
for item, candidates in rules_mapping.items():
    r.set(item, json.dumps(candidates))

print("Redis Feature Store populated!")

Redis Feature Store populated!


In [ ]:
from fastapi import FastAPI
import nest_asyncio
import uvicorn
import threading

app = FastAPI()
nest_asyncio.apply()

# Load mapping and model
with open('item_map.json', 'r') as f:
    item_to_id = json.load(f)
model = torch.load("bst_model.pt", weights_only=False)

@app.post("/recommend")
async def recommend(cart: list[str]):
    last_item = cart[-1]
    candidates_raw = r.get(last_item)

    if not candidates_raw:
        return {"recommendations": ["Coke", "Fries"]} # Fallback

    candidates = json.loads(candidates_raw)

    # Simple Ranking: Sort by Apriori Lift
    ranked = sorted(candidates, key=lambda x: x['score'], reverse=True)
    return {"recommendations": [c['item'] for c in ranked[:3]]}

# Run Server
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=start_server, daemon=True).start()

INFO:     Started server process [4330]
INFO:     Waiting for application startup.


In [ ]:
from fastapi import FastAPI
import nest_asyncio
import uvicorn
import threading

app = FastAPI()
nest_asyncio.apply()

# Load mapping and model
with open('item_map.json', 'r') as f:
    item_to_id = json.load(f)
model = torch.load("bst_model.pt", weights_only=False)

@app.post("/recommend")
async def recommend(cart: list[str]):
    last_item = cart[-1]
    candidates_raw = r.get(last_item)

    if not candidates_raw:
        return {"recommendations": ["Coke", "Fries"]}

    candidates = json.loads(candidates_raw)


    ranked = sorted(candidates, key=lambda x: x['score'], reverse=True)
    return {"recommendations": [c['item'] for c in ranked[:3]]}

# Run Server
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=start_server, daemon=True).start()

INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Started server process [4330]
INFO:     Waiting for application startup.


In [ ]:
import requests
import time

start = time.time()
response = requests.post("http://127.0.0.1:8000/recommend", json=["Chicken Biryani"])
latency = (time.time() - start) * 1000

print(f"Latency: {latency:.2f}ms")
print("Recommendations:", response.json())

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:42480 - "POST /recommend HTTP/1.1" 200 OK
Latency: 17.26ms
Recommendations: {'recommendations': ['Raita', 'Salan']}


In [ ]:
import logging
import warnings

# Hide the background noise
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("uvicorn.access").setLevel(logging.ERROR)

In [ ]:
result = requests.post("http://127.0.0.1:8000/recommend", json=["Chicken Biryani"]).json()
print(f"RESULT: {result}")

RESULT: {'recommendations': ['Raita', 'Salan']}


In [ ]:
# Display the first 5 rows
print("--- CSAO Rail: Synthetic Dataset Preview (First 5 Rows) ---")
display(df_synthetic.head())
print(f"\nDataset Dimensions: {df_synthetic.shape[0]} Rows x {df_synthetic.shape[1]} Columns")
print(f"Features: {', '.join(df_synthetic.columns)}")

--- CSAO Rail: Synthetic Dataset Preview (First 5 Rows) ---


,Order_ID,User_ID,Item_Name,Category,Quantity,Time_of_Day
0,1001,U542,Masala Dosa,Main,1,Morning
1,1002,U916,Veg Burger,Main,1,Evening
2,1003,U750,Margherita Pizza,Main,1,Late-night
3,1003,U750,Garlic Bread,Side,1,Late-night
4,1004,U544,Chicken Biryani,Main,1,Lunch



Dataset Dimensions: 726 Rows x 6 Columns
Features: Order_ID, User_ID, Item_Name, Category, Quantity, Time_of_Day
